# SLAM-ASR: Türkçe Mozilla Common Voice Konuşma Tanıma
### Mozilla Data Collective (MDC) & Modern Türkçe Çok Modlu ASR

> **Referans Makale:** *An Embarrassingly Simple Approach for LLM with Strong ASR Capacity* ([arXiv:2402.08846](https://arxiv.org/abs/2402.08846))
> **Veri Seti:** Mozilla Common Voice (Turkish) via [Mozilla Data Collective](https://mozilladatacollective.com)
> **MDC Dataset ID:** `cmu5wkah500c2o10719lem8m8`

Bu notebook, Google Colab (ücretsiz T4 veya ücretli A100/L4) üzerinde **Mozilla Common Voice Türkçe** veri seti ile SLAM-ASR modelini kurmak, eğitmek ve test etmek için hazırlanmıştır.

---
### Temel Mimarisi:
* **Akustik Kodlayıcı:** `openai/whisper-small` - **Dondurulmuş (Frozen)**
* **Dil Modeli (LLM):** `Qwen/Qwen2.5-7B-Instruct` - **Dondurulmuş (Frozen)**
* **Projektör:** $k=5$ Ardışık Çerçeve Birleştirme (50 Hz $\to$ 10 Hz) + Tek Lineer Katman - **Eğitilen Tek Bileşen** (~%0.6 parametre)

## 1. GPU ve Donanım Kontrolü
Colab oturumunuzun GPU hızlandırıcıya bağlı olduğunu doğrulayın (Menü: `Runtime` -> `Change runtime type` -> `T4 GPU` veya `A100 GPU`).

In [ ]:
# GPU kullanılabilirliğini kontrol edin
!nvidia-smi

import torch
print(f"[*] PyTorch Versiyonu: {torch.__version__}")
print(f"[*] CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")


Thu Oct 15 12:05:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Depo Dosyalarını Hazırlama ve Bağımlılıkları Yükleme
GitHub reposunu çekin veya çalışma dizinine geçin ve gerekli paketleri yükleyin.

In [ ]:
import os

# Eğer doğrudan GitHub'dan klonlayacaksanız:
if not os.path.exists("tr") and not os.path.exists("slam_asr"):
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement/tr/cv
elif os.path.exists("tr/cv"):
    %cd tr/cv

# Bağımlılıkları yükleyin
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio bitsandbytes accelerate datacollective mutagen


Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 945, done.
remote: Counting objects: 100% (945/945), done.
remote: Compressing objects: 100% (791/791), done.
remote: Total 945 (delta 222), reused 860 (delta 137), pack-reused 0 (from 0)
Receiving objects: 100% (945/945), 5.66 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (222/222), done.
/content/slam-asr-reimplement/tr/cv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 178.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires reque

## 3. Kod Dosyalarının Doğrulanması ve Birim Testler
Birim testleri koşturarak modüllerin ve Common Voice veri işleme mantığının hazır olduğunu doğrulayın.

In [ ]:
!python run_tests.py


test_variable_length_waveform_collation (test_collator.TestCollator.test_variable_length_waveform_collation) ... ok
test_whisper_mel_collation (test_collator.TestCollator.test_whisper_mel_collation) ... ok
test_a100_40gb_tier_recommendations (test_device_utils.TestDeviceUtils.test_a100_40gb_tier_recommendations) ... ok
test_cpu_tier_recommendations (test_device_utils.TestDeviceUtils.test_cpu_tier_recommendations) ... ok
test_get_device_specs (test_device_utils.TestDeviceUtils.test_get_device_specs) ... ok
test_h100_tier_recommendations (test_device_utils.TestDeviceUtils.test_h100_tier_recommendations) ... ok
test_l4_tier_recommendations (test_device_utils.TestDeviceUtils.test_l4_tier_recommendations) ... ok
test_mps_tier_recommendations (test_device_utils.TestDeviceUtils.test_mps_tier_recommendations) ... ok
test_resolve_auto_hyperparameters (test_device_utils.TestDeviceUtils.test_resolve_auto_hyperparameters) ... ok
test_rtx_a4000_tier_recommendations (test_device_utils.TestDeviceUtil

## 4. Hızlı Doğrulama: Sentetik Türkçe Veri ile Test Eğitimi (30 Saniye)
Büyük veri setini indirmeden önce tüm boru hattının sorunsuz çalıştığını sentetik verimizle test edin.

In [ ]:
# 1. Sentetik Türkçe test verisini üretin
!python scripts/create_toy_turkish_dataset.py

# 2. Hızlı doğrulama eğitimini koşturun (50 adım)
!python train.py --config config/train_toy_debug_turkish.yaml


[*] Generating synthetic Turkish toy dataset...
[*] Created 8 train samples in: /content/slam-asr-reimplement/tr/cv/data/sample_train_tr.jsonl
[*] Created 2 val samples in:   /content/slam-asr-reimplement/tr/cv/data/sample_val_tr.jsonl
[*] Turkish toy dataset generated successfully.
[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       float32
      • load_in_4bit:      False (NF4)
      • batch_size:        2
      • grad_accumulation: 1  --> Effective Batch: 2
      • mixed_precision:   bf16=False, fp16=False
      • num_workers:       0
[*] Initializing SLAM-ASR with config: config/train_toy_debug_turkish.yaml


## 5. Mozilla Common Voice (Türkçe) Veri Seti İndirme ve Hazırlığı
Colab Secrets (Sol paneldeki anahtar simgesi 🔑) üzerinden `MDC_API_KEY` anahtarınız okunur ve Mozilla Data Collective API'si aracılığıyla dataset (`cmu5wkah500c2o10719lem8m8`) otomatik olarak indirilip hazırlanır.

In [ ]:
from google.colab import userdata
import os

# Colab Secrets'tan MDC_API_KEY anahtarını alın
try:
    mdc_api_key = userdata.get('MDC_API_KEY')
    print("[*] MDC_API_KEY anahtari Colab Secrets uzerinden basariyla alindi.")
except Exception:
    mdc_api_key = input("Lutfen Mozilla Data Collective API anahtarinizi (MDC_API_KEY) girin: ")

# Mozilla Common Voice Türkçe veri setini indirin ve hazırlayın
!python scripts/prepare_common_voice.py \
    --download \
    --api_key "{mdc_api_key}" \
    --output_dir data/cv


[*] MDC_API_KEY anahtari Colab Secrets uzerinden basariyla alindi.
[*] Connecting to Mozilla Data Collective REST API...
[*] Requesting presigned download URL from Mozilla Data Collective API...
    URL: https://mozilladatacollective.com/api/datasets/cmu5wkah500c2o10719lem8m8/download
[*] Downloading dataset to: /content/slam-asr-reimplement/tr/cv/data/cv/common-voice-scripted-speech-27-0-turkis-b14760fe.tar.gz
    Progress: 0% (8.0 MB / 2853.8 MB)
    Progress: 5% (144.0 MB / 2853.8 MB)
    Progress: 10% (288.0 MB / 2853.8 MB)
    Progress: 15% (432.0 MB / 2853.8 MB)
    Progress: 20% (576.0 MB / 2853.8 MB)
    Progress: 25% (720.0 MB / 2853.8 MB)
    Progress: 30% (864.0 MB / 2853.8 MB)
    Progress: 35% (1000.0 MB / 2853.8 MB)
    Progress: 40% (1144.0 MB / 2853.8 MB)
    Progress: 45% (1288.0 MB / 2853.8 MB)
    Progress: 50% (1432.0 MB / 2853.8 MB)
    Progress: 55% (1576.0 MB / 2853.8 MB)
    Progress: 60% (1720.0 MB / 2853.8 MB)
    Progress: 65% (1856.0 MB / 2853.8 MB)
    Prog

## 6. Donanıma Göre Otomatik Optimize Olan SLAM-ASR Türkçe Eğitimi
Donanım profilleyicimiz (`device_utils`) GPU'nuza göre en verimli hassasiyeti (bfloat16/float16), kuantizasyonu (4-bit NF4) ve batch boyutunu otomatik olarak belirler.

In [ ]:
# Donanıma göre otomatik optimize edilen Türkçe SLAM-ASR eğitimi
!python train.py --config config/hardware_adaptive_turkish.yaml


[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       bfloat16
      • load_in_4bit:      False (NF4)
      • batch_size:        16
      • grad_accumulation: 1  --> Effective Batch: 16
      • mixed_precision:   bf16=True, fp16=False
      • num_workers:       4
[*] Initializing SLAM-ASR with config: config/hardware_adaptive_turkish.yaml
config.json: 100% 1.97k/1.97k [00:00<00:00, 15.6MB/s]

model.safetensors: downloading bytes:  19% 182M/967M [00:01<00:03, 260MB/s, 13.9MB/s  ]
model.safetensors: reconstructing file:  29% 277M/967M [00:01<00:03, 229MB/s]
model.safetensors: downloading bytes:  25% 238M/967M [00:01

## 7. Model Değerlendirmesi (Türkçe WER ve CER Hesaplama)
Eğitilen modelin Common Voice test seti üzerindeki performansını ölçün. Türkçe büyük/küçük harf (`İ -> i`, `I -> ı`) ve noktalama normalizasyonu otomatik uygulanır.

In [ ]:
!python evaluate.py \
    --config config/hardware_adaptive_turkish.yaml \
    --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint \
    --manifest data/cv/test.jsonl \
    --language tr \
    --beam_size 4


[*] Loading model on cuda...
Loading weights: 100% 479/479 [00:00<00:00, 14602.72it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:00<00:00, 14850.58it/s]
[*] Loading projector weights from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt
[*] Evaluating 11874 samples with beam_size=4...
Evaluating:   0% 0/743 [00:00<?, ?it/s]/content/slam-asr-reimplement/tr/cv/evaluate.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=amp_dtype):
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1463: UserWarning: Passing `no_

## 8. İnteraktif Çıkarım (Test Sesini Dinleme ve Transkribe Etme)
Test setindeki bir Common Voice ses kaydını dinleyin ve modelinizin konuşmayı nasıl yazıya döktüğünü canlı olarak görün.

In [ ]:
import json
import os
import IPython.display as ipd

# Test setinden ilk örneği bulun
sample_audio = None
sample_text = ""

manifest_file = "data/cv/test.jsonl" if os.path.exists("data/cv/test.jsonl") else "data/sample_val_tr.jsonl"
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        if first_line:
            data = json.loads(first_line)
            sample_audio = data.get("audio_filepath")
            sample_text = data.get("text", "")

if sample_audio and os.path.exists(sample_audio):
    print(f"[*] Seçilen Ses: {sample_audio}")
    print(f"[*] Gerçek Metin (Ground Truth): {sample_text}\n")
    display(ipd.Audio(sample_audio))

    !python inference.py \
        --audio "{sample_audio}" \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt \
        --prompt "Konuşmayı metne dönüştürün."
else:
    print("[!] Test ses dosyası bulunamadı. Lütfen ses dosyasının yolunu elle belirtin.")


[*] Seçilen Ses: /content/slam-asr-reimplement/tr/cv/data/cv/extracted/cv-corpus-27.0-2026-09-11/tr/clips/common_voice_tr_21461823.mp3
[*] Gerçek Metin (Ground Truth): belgrad da aynı derecede kuşkulu görünüyor



[*] Loading SLAM-ASR on cuda...
Loading weights: 100% 479/479 [00:00<00:00, 14663.26it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:00<00:00, 14646.06it/s]
[*] Loading projector weights from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt
[*] Processing audio: /content/slam-asr-reimplement/tr/cv/data/cv/extracted/cv-corpus-27.0-2026-09-11/tr/clips/common_voice_tr_21461823.mp3
[*] Transcribing (beam_size=4)...
/content/slam-asr-reimplement/tr/cv/inference.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=amp_dtype):
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/

## 9. Zaman Damgalı Google Drive Yedeklemesi
Eğitilen ağırlıkları (`best_checkpoint`), test sonuçlarını (`eval_results.jsonl`), manifestoları ve konfigürasyonu Google Drive'a tarih/saat damgalı olarak kalıcı yedekleyin.

In [ ]:
# 9. Adım: Tüm Model Ağırlıklarını, Manifestoları ve Test Sonuçlarını Drive'a Yedekleme
import os
from datetime import datetime
from google.colab import drive

# 1. Google Drive'ı bağlayın
drive.mount("/content/drive")

# 2. Zaman damgalı hedef klasörü belirleyin
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_target_dir = f"/content/drive/MyDrive/checkpoints/slam_asr_cv_turkish_{timestamp}"
os.makedirs(drive_target_dir, exist_ok=True)

# 3. Model Ağırlıklarını Kopyalayın
!cp -r outputs/slam_asr_cv_turkish_adaptive/best_checkpoint "{drive_target_dir}/"
!cp -r outputs/slam_asr_cv_turkish_adaptive/final_checkpoint "{drive_target_dir}/"

# 4. (Varsa) Değerlendirme Çıktısını Kopyalayın
if os.path.exists("eval_results.jsonl"):
    !cp eval_results.jsonl "{drive_target_dir}/"

# 5. Veri Manifestolarını Kopyalayın (train/val/test jsonl)
if os.path.exists("data/cv"):
    os.makedirs(f"{drive_target_dir}/manifests", exist_ok=True)
    !cp data/cv/*.jsonl "{drive_target_dir}/manifests/"

# 6. Kullanılan Config Dosyasını Kopyalayın
!cp config/hardware_adaptive_turkish.yaml "{drive_target_dir}/"

print("\n" + "=" * 65)
print(f"[*] Tam Yedekleme Başarıyla Tamamlandı!")
print(f"[*] Hedef Dizin: {drive_target_dir}")
print("    • best_checkpoint & final_checkpoint")
print("    • eval_results.jsonl (Test tahminleri ve WER analizi)")
print("    • train/val/test manifestoları")
print("    • hardware_adaptive_turkish.yaml konfigürasyonu")
print("=" * 65)


## 10. Modeli Belleğe Yükle ve Canlı Mikrofon Kaydı ile Konuşmayı Metne Dönüştür
Model GPU belleğine bir kere yüklenir ve hazır bekler. Ardından tarayıcınızdan mikrofona Türkçe konuşursunuz ve model konuşmanızı 1 saniye içinde yazıya döker.

In [ ]:
# 10. Adım: Modeli Belleğe Yükle ve Mikrofon ile Canlı Konuşmayı Metne Dönüştür
import os
import glob
import subprocess
import yaml
from base64 import b64decode
import torch
import IPython.display as ipd
from google.colab import output

from slam_asr.models.slam_model import SLAM_ASR, SLAMASRConfig
from slam_asr.data.audio_processor import AudioProcessor
from slam_asr.utils.device_utils import resolve_auto_hyperparameters

# 1. Drive'ı Bağla ve En Güncel Checkpoint'i Bul
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_cv_turkish_*/best_checkpoint/projector.pt"))
if not checkpoint_candidates:
    checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_turkish_*/best_checkpoint/projector.pt"))

if checkpoint_candidates:
    best_ckpt = checkpoint_candidates[-1]
elif os.path.exists("outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"):
    best_ckpt = "outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"
else:
    raise FileNotFoundError("Hiçbir best_checkpoint/projector.pt dosyası bulunamadı!")

# 2. Modeli GPU'ya Yükle (Zaten Yüklüyse Tekrar Yüklemez!)
if "loaded_slam_model" not in globals():
    print(f"[*] Model checkpoint'i kullanılıyor:\n    {best_ckpt}\n")
    print("[*] 1/3: Model GPU VRAM'ine yükleniyor... Lütfen bekleyin...")

    cfg = yaml.safe_load(open("config/hardware_adaptive_turkish.yaml"))
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    model_cfg, data_cfg, _ = resolve_auto_hyperparameters(model_cfg, data_cfg, {})

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_config = SLAMASRConfig(**model_cfg)

    model = SLAM_ASR(config=model_config)
    if not getattr(model_config, "load_in_4bit", False) and not getattr(model_config, "load_in_8bit", False):
        model = model.to(device)
    else:
        model.speech_encoder = model.speech_encoder.to(device)
        model.projector = model.projector.to(device)

    state_dict = torch.load(best_ckpt, map_location=device)
    model.projector.load_state_dict(state_dict)
    model.eval()

    audio_proc = AudioProcessor(
        sampling_rate=data_cfg.get("sampling_rate", 16000),
        is_whisper=data_cfg.get("is_whisper", True),
        whisper_mel_bins=data_cfg.get("whisper_mel_bins", 80),
    )

    loaded_slam_model = model
    loaded_audio_proc = audio_proc
    loaded_device = device
    loaded_model_config = model_config
    print("[*] 2/3: Model başarıyla VRAM'e yüklendi ve hazır bekliyor! 🚀\n")
else:
    print("[*] Model zaten VRAM'de hazır bekliyor (Yeniden yükleme yapılmadı). 🚀\n")

# 3. Model Hazır Olduktan Sonra Mikrofon Butonunu Göster
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result);
  reader.readAsDataURL(blob);
});
var recordAudio = () => new Promise(async (resolve, reject) => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);

    const container = document.createElement("div");
    container.style.cssText = "padding: 12px; margin: 10px 0; background: #1e1e2e; border-radius: 8px; color: white; font-family: sans-serif; display: inline-block;";

    const btn = document.createElement("button");
    btn.innerHTML = "🔴 Kaydı Başlat (Konuşmaya Başlayın)";
    btn.style.cssText = "padding: 12px 24px; font-size: 16px; font-weight: bold; background: #e74c3c; color: white; border: none; border-radius: 6px; cursor: pointer; transition: 0.3s; box-shadow: 0 4px 6px rgba(0,0,0,0.3);";
    container.appendChild(btn);
    document.body.appendChild(container);

    let isRecording = false;

    btn.onclick = () => {
      if (!isRecording) {
        recorder.start();
        isRecording = true;
        btn.innerHTML = "⏹️ Kaydı Bitir (Sonucu Gör)";
        btn.style.background = "#27ae60";
      } else {
        recorder.stop();
        btn.innerHTML = "⏳ Model transkribe ediyor...";
        btn.disabled = true;
        btn.style.background = "#7f8c8d";
      }
    };

    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: "audio/webm" });
      const base64Text = await b2text(blob);
      container.remove();
      resolve(base64Text);
    };
  } catch (err) {
    reject(err);
  }
});
recordAudio();
"""

print("[*] 3/3: Mikrofon hazır. Konuşmak için aşağıdaki butona basın:")
audio_data = output.eval_js(RECORD_JS)

# 4. Sesi 16 kHz Mono WAV Formatına Dönüştür
webm_file = "live_temp.webm"
wav_file = "live_mic_recording.wav"

with open(webm_file, "wb") as f:
    f.write(b64decode(audio_data.split(",")[1]))

subprocess.run([
    "ffmpeg", "-y", "-i", webm_file,
    "-ar", "16000", "-ac", "1",
    wav_file
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n[*] Kaydettiğiniz Ses:")
display(ipd.Audio(wav_file))

# 5. Bellekteki Model ile Anında Çıkarım
speech_tensor = loaded_audio_proc.process(wav_file).unsqueeze(0).to(loaded_device)
use_amp = (loaded_model_config.torch_dtype in ("float16", "bfloat16")) and loaded_device.type == "cuda"
amp_dtype = torch.bfloat16 if loaded_model_config.torch_dtype == "bfloat16" else torch.float16

with torch.no_grad():
    if use_amp:
        with torch.cuda.amp.autocast(dtype=amp_dtype):
            transcription = loaded_slam_model.generate(
                speech_inputs=speech_tensor,
                prompts=["Konuşmayı metne dönüştürün."],
                num_beams=4,
            )[0]
    else:
        transcription = loaded_slam_model.generate(
            speech_inputs=speech_tensor,
            prompts=["Konuşmayı metne dönüştürün."],
            num_beams=4,
        )[0]

print("\n" + "=" * 55)
print(f"🗣️  Sizin Konuşmanız: {transcription}")
print("=" * 55)
